In [1]:
from physics.simulation import mcfm, msq
from physics.hzz import zpair, zz4l
from physics.hstar import c6

import numpy as np
import json
import hist
import matplotlib, matplotlib.pyplot as plt
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "text.usetex": True,
    "pgf.texsystem": "lualatex",
    "pgf.rcfonts": False,
    "pgf.preamble": "", 
})

In [ ]:
with open('data/xsec.json') as f:
    xsec = json.load(f)

xsec = {
    msq.Component.SBI : xsec['gg4l_sbi'],
    msq.Component.int : xsec['gg4l_int'],
    msq.Component.INT : xsec['gg4l_int'],
    msq.Component.BKG : xsec['gg4l_bkg']
}

filenames = {
    msq.Component.SBI : 'data/ggZZ2e2m_sbi/events.csv',
    msq.Component.int : 'data/ggZZ2e2m_int/events.csv',
    msq.Component.INT : 'data/ggZZ2e2m_int/events.csv',
    msq.Component.BKG : 'data/ggZZ2e2m_bkg/events.csv'
}

component_names = {
    msq.Component.SBI : 'SBI',
    msq.Component.int : 'SIG',
    msq.Component.INT : 'INT',
    msq.Component.BKG : 'BKG'
}

In [3]:
events_sbi = mcfm.from_csv(cross_section=xsec[msq.Component.SBI], file_path=filenames[msq.Component.SBI], n_rows=1000)
        
z_cand = zpair.ZPairCandidate(algorithm='leastsquare')
z_masses = zpair.ZPairMassWindow(z1=(70,115), z2=(70,115))
mandelstam = zz4l.MandelstamVariables()
four_lepton = zz4l.FourLeptonSystem()

events_sbi = events_sbi.calculate(z_cand).filter(z_masses).calculate(four_lepton)
events_sig = events_sbi.reweight(msq.Component.SBI, msq.Component.SIG)
events_int = events_sbi.reweight(msq.Component.SBI, msq.Component.INT)

In [30]:
c6_vals = np.linspace(-20, 20, 101)
c6_colors = ['red', 'blue']
c6_lines = ['-', '-']

mod_sig_c6 = c6.Modifier(baseline=msq.Component.SIG, events=events_sig, c6_values=[-5,-1,0,1,5])
mod_int_c6 = c6.Modifier(baseline=msq.Component.INT, events=events_int, c6_values=[-5,0,5])
mod_sbi_c6 = c6.Modifier(baseline=msq.Component.SBI, events=events_sbi, c6_values=[-5,-1,0,1,5])

wt_sig_c6, prob_sig_c6 = mod_sig_c6.modify(c6_vals)
wt_int_c6, prob_int_c6 = mod_int_c6.modify(c6_vals)
wt_sbi_c6, prob_sbi_c6 = mod_sbi_c6.modify(c6_vals)

In [31]:
sig_bsm_over_sm = wt_sig_c6 / events_sig.weights.to_numpy()[:,np.newaxis]
int_bsm_over_sm = wt_int_c6 / events_int.weights.to_numpy()[:,np.newaxis]
sbi_bsm_over_sm = wt_sbi_c6 / events_sbi.weights.to_numpy()[:,np.newaxis]

In [32]:
event_index = 100

In [34]:
fig, (ax_sig, ax_int, ax_sbi) = plt.subplots(3,1, gridspec_kw={'height_ratios': [1,1,1]}, figsize=(6,6), sharex=True)

# ax_sig.plot(c6_vals, sig_bsm_over_sm[90], linestyle='--')
# ax_int.plot(c6_vals, int_bsm_over_sm[90], linestyle='--')
# ax_sbi.plot(c6_vals, sbi_bsm_over_sm[90], linestyle='--')

msq_sig_c6 = wt_sig_c6 / events_sig.weights.to_numpy()[:,np.newaxis] * events_sig.components['msq_sig_sm'].to_numpy()[:,np.newaxis]
msq_int_c6 = wt_int_c6 / events_int.weights.to_numpy()[:,np.newaxis] * events_int.components['msq_int_sm'].to_numpy()[:,np.newaxis]
msq_sbi_c6 = wt_sbi_c6 / events_sbi.weights.to_numpy()[:,np.newaxis] * events_sbi.components['msq_sbi_sm'].to_numpy()[:,np.newaxis]

c6_pts = []
msq_sig_pts = []
msq_int_pts = []
msq_sbi_pts = []
for c6_val, msq_name in mcfm.mcfm_component_c6[msq.Component.SBI].items():
    c6_pts.append(c6_val)
    msq_sbi_pts.append(events_sbi.components[msq_name][event_index])
for msq_name in mcfm.mcfm_component_c6[msq.Component.SIG].values():
    msq_sig_pts.append(events_sig.components[msq_name][event_index])
for msq_name in mcfm.mcfm_component_c6[msq.Component.INT].values():
    msq_int_pts.append(events_int.components[msq_name][event_index])

ax_sig.plot(c6_vals, msq_sig_c6[event_index], linestyle='--', color='blue')
ax_int.plot(c6_vals, msq_int_c6[event_index], linestyle='--', color='blue')
ax_sbi.plot(c6_vals, msq_sbi_c6[event_index], linestyle='--', color='blue')

ax_sig.scatter(c6_pts, msq_sig_pts, color='black')
ax_int.scatter(c6_pts, msq_int_pts, color='black')
ax_sbi.scatter(c6_pts, msq_sbi_pts, color='black')

ax_sig.set_ylabel('$|\\mathcal{M}_{gg \\rightarrow h^{\\ast} \\rightarrow ZZ}|^2$')
ax_int.set_ylabel('$2\\mathrm{Re}(\\mathcal{M}^{\\dag}_{gg \\rightarrow h^{\\ast} \\rightarrow ZZ} \\mathcal{M}_{gg \\rightarrow ZZ})$')
ax_sbi.set_ylabel('$|\\mathcal{M}_{gg (\\rightarrow h^{\\ast}) \\rightarrow ZZ}|^2$')

ax_sbi.set_xlabel('$c_6$')

plt.tight_layout()
plt.savefig('msq_bsm_over_sm.pdf')
plt.close()